# Hyperparameter Tuning Lesson (7/21/26)

---

*Codecademy — Deep Learning with TensorFlow Path: Hyperparameter Tuning. Concise grad-student notes.*

## Introduction to Hyperparameter Tuning

Building a neural network doesn't end at "train it once." We previously picked values for **hyperparameters** — settings chosen *before* training that aren't learned from data:

- learning rate
- number of batches (batch size)
- number of epochs
- number of units per hidden layer
- activation functions

Initial choices are often a "hunch" based on common practice, then refined by experimentation.

### The ML pipeline: three data splits

- **Training set** — fits the model by updating weights/biases. (Previously: 67% of data.)
- **Validation set** — checks the model's fit as hyperparameters are tuned. This is a *biased* evaluation, since we adjust the model based on it.
- **Test set** — a *final, unbiased* evaluation used to compare models. **Never used for hyperparameter tuning** — only to report final results (e.g. a Kaggle leaderboard).

A common split is **70% train / 20% validation / 10% test**, though the right ratio depends on data volume and task. (Previously we only used a 67/33 train/test split, with no validation set.)

### The tuning loop

1. Set hyperparameters to initial values.
2. Fit the model on the **training** set.
3. Evaluate on the **validation** set (accuracy for classification, MAE for regression).
4. If performance is unsatisfactory, tweak hyperparameters — learning rate, batch size, epochs, number of hidden layers, optimizer — and repeat from step 2.
5. Once satisfied, do a final evaluation on the **test** set and report results.

### Pipeline diagram

```
Hyperparameters              Training              Validation           Performance?
(lr: 0.1, batch: 8,     ->  Model fit (weights  ->  Performance    ->   Good -> Evaluate on Test
 epochs: 200,                + bias)                 evaluation          Bad  -> change hyperparameters,
 hidden layers: 1,                                                              loop back to Model fit
 optimizer: adam)
```

Concretely: pick a hyperparameter set -> fit weights/bias on **Training** data -> evaluate on **Validation** data -> if performance is bad, change the hyperparameters and refit; if good, run the model once on **Test** data for the final, unbiased evaluation.

## Q & A

**Q: What is an epoch and/or batch?**

Training data is fed to the model in chunks called **batches**, rather than all at once.

- **Batch size** — the number of training examples processed before the model updates its weights once. E.g. `batch_size=32` means the model looks at 32 samples, computes the average error, then does one gradient-descent update.
- **Epoch** — one full pass through the *entire* training set. If you have 1,000 samples and `batch_size=100`, one epoch consists of 10 batches (10 weight updates).

So `epochs` controls how many times the model sees the whole dataset, and `batch_size` controls how many samples it sees between each weight update *within* an epoch. Smaller batches mean noisier but more frequent updates; larger batches mean smoother but less frequent updates (and more memory use per step).

**Q: What does `verbose` mean again?**

`verbose` controls how much progress output `.fit()` (or `.evaluate()`/`.predict()`) prints during training — it has no effect on training itself, only on logging.

- `verbose=0` — silent, no output.
- `verbose=1` — a progress bar per batch, updated live within each epoch (the default, and most common for interactive work).
- `verbose=2` — one line printed per epoch, no per-batch progress bar (better for logging to a file).

## Using a Validation Set for Hyperparameter Tuning

Choosing hyperparameters based on training-set performance risks **overfitting**: the model (and our hyperparameter choices) latch onto patterns specific to the training data that don't generalize. So hyperparameters are tuned against a held-out **validation set** instead.

In Keras, `.fit()` can carve out a validation split automatically via `validation_split`:

In [1]:
# my_model.fit(data, labels, epochs=20, batch_size=1, verbose=1, validation_split=0.2)
#
# validation_split: float in [0, 1] -- fraction of the TRAINING data held out as validation data.
# Here, 20% of `data`/`labels` is set aside: the model never trains on it, and at the end of
# each epoch its loss (and any metrics) are evaluated on that held-out slice.
# Usually a small fraction (e.g. 0.1-0.2) of the training data.

### Exercise: fit with a validation split

Fit `model` on `features_train`/`labels_train` with 40 epochs, batch size 8, verbose on, and a 33% validation split.

In [2]:
# from model import design_model, features_train, labels_train
#
# model = design_model(features_train, learning_rate=0.01)
# model.fit(features_train, labels_train, epochs=40, batch_size=8, verbose=1, validation_split=0.33)

## Manual Tuning: Learning Rate

Networks train via **gradient descent**; the **learning rate** sets how big a weight update is applied for a given error gradient computed on a batch.

- **Too large** — faster learning, but risks getting stuck in a suboptimal (local-minimum) solution, or even an unstable, oscillating loss that never settles.
- **Too small** — can reach a good (even global) solution, but converges very slowly, or may fail to converge / get stuck in a local minimum within the allotted training time.

Rough intuition from example learning rates:

| Learning rate | Behavior |
|---|---|
| `1.0` | oscillates — too large, unstable |
| `0.01` | good performance — reasonable middle ground |
| `1e-07` | too small — almost nothing learned in the time given |

Learning rate is usually one of the first hyperparameters worth sweeping manually.

### Exercise: comparing learning rates via loss curves

Fit the same model architecture at several learning rates and plot each run's training loss (blue) vs. validation loss (orange) side by side, to see the too-large / good / too-small behavior directly. Tried `learning_rates = [1e-3, 1e-4, 1e-7]` with `epochs=100`, `batch_size=10` (fixed across runs, only `learning_rate` varies).

In [3]:
# from model import design_model, features_train, labels_train
# import matplotlib.pyplot as plt
#
# def fit_model(f_train, l_train, learning_rate, num_epochs, bs):
#     model = design_model(f_train, learning_rate)
#     history = model.fit(f_train, l_train, epochs=num_epochs, batch_size=bs, verbose=0, validation_split=0.2)
#     # plot learning curves
#     plt.plot(history.history['loss'], label='train')
#     plt.plot(history.history['val_loss'], label='validation')
#     plt.title('lrate=' + str(learning_rate))
#     plt.legend(loc="upper right")
#
# learning_rates = [1E-3, 1E-4, 1E-7]
# num_epochs = 100
# batch_size = 10
#
# for i in range(len(learning_rates)):
#     plot_no = 420 + (i + 1)
#     plt.subplot(plot_no)
#     fit_model(features_train, labels_train, learning_rates[i], num_epochs, batch_size)
#
# plt.tight_layout()
# plt.show()

# TODO: which of 1e-3 / 1e-4 / 1e-7 gave the best performance on your run? (see the saved plot)

## Manual Tuning: Batch Size

**Batch size** = how many training samples are seen before the model updates its weights/bias.

- **Batch gradient descent** — batch size = entire training set.
- **Stochastic gradient descent (SGD)** — batch size = 1.
- **Mini-batch gradient descent** — 1 < batch size < training set size (the common case).

Batching also enables GPU parallelization of the computation.

### Tradeoff

- **Larger batch size** — better gradient estimates, closer to the true optimum, but more expensive computationally and can generalize worse.
- **Smaller batch size** — noisier gradient estimates, but faster learning (more updates per epoch).

The right size is dataset/problem-dependent and found via tuning. Example sweep: fix `learning_rate=0.01`, try batch sizes `1, 2, 10, 16` — smaller batch sizes show higher variance (more oscillation) in the loss curve.

**Trick:** if you increase the batch size, also increase the learning rate to help recover the performance lost from noisier-but-larger gradient steps.

### Exercise: comparing batch sizes, then compensating with a higher learning rate

1. Try `batches = [4, 32, 64]` at the original (smaller) learning rate — larger batches (32, 64) perform worse.
2. Raise `learning_rate` to `0.1` and rerun — larger batches recover, illustrating the "bump the learning rate when you bump the batch size" trick from above.

In [4]:
# from model import features_train, labels_train, design_model
# import matplotlib.pyplot as plt
#
# def fit_model(f_train, l_train, learning_rate, num_epochs, batch_size, ax):
#     model = design_model(features_train, learning_rate)
#     history = model.fit(features_train, labels_train, epochs=num_epochs, batch_size=batch_size,
#                          verbose=0, validation_split=0.3)
#     # plot learning curves
#     ax.plot(history.history['mae'], label='train')
#     ax.plot(history.history['val_mae'], label='validation')
#     ax.set_title('batch = ' + str(batch_size), fontdict={'fontsize': 8, 'fontweight': 'medium'})
#     ax.set_xlabel('# epochs')
#     ax.set_ylabel('mae')
#     ax.legend()
#
# learning_rate = 0.1   # checkpoint 2: raised from a smaller value after seeing bad performance at batch 32/64
# num_epochs = 100
# batches = [4, 32, 64]  # checkpoint 1
#
# fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex='col', sharey='row',
#                                      gridspec_kw={'hspace': 0.7, 'wspace': 0.4})
# axes = [ax1, ax2, ax3]
#
# for i in range(len(batches)):
#     fit_model(features_train, labels_train, learning_rate, num_epochs, batches[i], axes[i])
#
# plt.savefig('static/images/my_plot.png')

# TODO: at learning_rate=0.1, does raising it actually fix batch=32/64 performance in your run?

## Manual Tuning: Epochs and Early Stopping

**Epochs** = number of complete passes through the training set (typically large: 100, 1000+). One epoch means the optimizer has seen every batch once.

- **Too many epochs** -> overfitting (model memorizes training data, validation performance degrades).
- **Too few epochs** -> underfitting (model hasn't learned enough).

**Early stopping**: stop training once the validation performance plateaus or starts degrading, instead of committing to a fixed epoch count up front.

Example: with an intentionally overparameterized model, training MAE ends around ~1000 but validation MAE ends around ~3034 -- a large train/val gap indicating overfitting. The validation curve shows it bottoms out and starts rising around epoch 50, meaning training past that point only hurts generalization.

### `EarlyStopping` callback (Keras)

- `monitor='val_loss'` -- which metric to watch.
- `mode='min'` -- stop when that metric stops decreasing (use `'max'` for metrics like accuracy).
- `patience=40` -- how many additional epochs to tolerate with no improvement before actually stopping (guards against stopping too early on a temporary plateau).

In [5]:
# from tensorflow.keras.callbacks import EarlyStopping
#
# stop = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=40)
#
# history = model.fit(features_train, labels_train, epochs=num_epochs, batch_size=16,
#                      verbose=0, validation_split=0.2, callbacks=[stop])

### Exercise: applying `EarlyStopping` in `fit_model()`

Inside `fit_model()`, create `es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)` just before `.fit()`, then pass it via `callbacks=[es]`. With `learning_rate=0.1`, `num_epochs=500` (capped -- early stopping should kick in well before then), lesson states early stopping should trigger at **epoch 47**.

In [6]:
# from model import features_train, labels_train, design_model
# import matplotlib.pyplot as plt
# from tensorflow.keras.callbacks import EarlyStopping
#
# def fit_model(f_train, l_train, learning_rate, num_epochs):
#     # model.py increases hidden neurons here to introduce some overfitting
#     model = design_model(features_train, learning_rate)
#     es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)
#     history = model.fit(features_train, labels_train, epochs=num_epochs, batch_size=16,
#                          verbose=0, validation_split=0.2, callbacks=[es])
#     return history
#
# learning_rate = 0.1
# num_epochs = 500
# history = fit_model(features_train, labels_train, learning_rate, num_epochs)
#
# fig, axs = plt.subplots(1, 2, gridspec_kw={'hspace': 1, 'wspace': 0.5})
# (ax1, ax2) = axs
# ax1.plot(history.history['loss'], label='train')
# ax1.plot(history.history['val_loss'], label='validation')
# ax1.set_title('lrate=' + str(learning_rate))
# ax1.legend(loc="upper right")
# ax1.set_xlabel("# of epochs")
# ax1.set_ylabel("loss (mse)")
#
# ax2.plot(history.history['mae'], label='train')
# ax2.plot(history.history['val_mae'], label='validation')
# ax2.set_title('lrate=' + str(learning_rate))
# ax2.legend(loc="upper right")
# ax2.set_xlabel("# of epochs")
# ax2.set_ylabel("MAE")
#
# print("Final training MAE:", history.history['mae'][-1])
# print("Final validation MAE:", history.history['val_mae'][-1])
#
# plt.savefig('static/images/my_plot.png')

# TODO: verify early stopping actually triggers at epoch 47 and check final train/val MAE in your run

## Manual Tuning: Changing the Model

Previously: a too-big model trained too long overfits. Here: the opposite failure -- a **too-simple model**.

Compared a one-layer model (`Dense(1)` directly on the input, no hidden layer) against a model with one hidden layer (`Dense(64, activation='relu')` -> `Dense(1)`).

- **One-layer model** -- validation curve stays *below* the training curve throughout, and performance is bad the whole time (no early stopping ever triggers, since there's no plateau/degradation to detect -- it just never gets good). This is **underfitting**: the model is too simple to capture the pattern, so more training doesn't help.
- **One-hidden-layer model** -- well-behaved learning curve, early stopping triggers at epoch 38, much better result overall.

### Choosing hidden layers / units

No definitive rule. Common heuristic: start with **one hidden layer**, with as many units as there are features in the dataset -- then iterate by observing the learning curves, since this heuristic doesn't always work.

In [7]:
# def one_layer_model(X, learning_rate):
#     ...
#     model.add(input)
#     model.add(layers.Dense(1))
#     ...
#
# def more_complex_model(X, learning_rate):
#     ...
#     model.add(input)
#     model.add(layers.Dense(64, activation='relu'))
#     model.add(layers.Dense(1))
#     ...

### Exercise: shrinking the hidden layer from 64 to 8 units

Full pipeline: `one_layer_model()` (no hidden layer) vs `more_complex_model()` (one hidden `Dense` layer, now with only 8 units instead of 64), both fit with `EarlyStopping(patience=20)`, `learning_rate=0.1`, `batch_size=2`, `num_epochs=200`.

In [8]:
# import tensorflow as tf
# from tensorflow.keras.models import Sequential
# from tensorflow.keras import layers
# from tensorflow.keras.callbacks import EarlyStopping
# import matplotlib.pyplot as plt
# from model import features_train, labels_train
#
# def more_complex_model(X, learning_rate):
#     model = Sequential(name="my_first_model")
#     input = tf.keras.Input(shape=(X.shape[1],))
#     model.add(input)
#     model.add(layers.Dense(8, activation='relu'))  # reduced from 64 -> 8
#     model.add(layers.Dense(1))
#     opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)
#     model.compile(loss='mse', metrics=['mae'], optimizer=opt)
#     return model
#
# def one_layer_model(X, learning_rate):
#     model = Sequential(name="my_first_model")
#     input = tf.keras.Input(shape=(X.shape[1],))
#     model.add(input)
#     model.add(layers.Dense(1))
#     opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)
#     model.compile(loss='mse', metrics=['mae'], optimizer=opt)
#     return model
#
# def fit_model(model, f_train, l_train, learning_rate, num_epochs):
#     es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)
#     history = model.fit(features_train, labels_train, epochs=num_epochs, batch_size=2,
#                          verbose=0, validation_split=0.2, callbacks=[es])
#     return history
#
# def plot(history):
#     fig, axs = plt.subplots(1, 2, gridspec_kw={'hspace': 1, 'wspace': 0.8})
#     (ax1, ax2) = axs
#     ax1.plot(history.history['loss'], label='train')
#     ax1.plot(history.history['val_loss'], label='validation')
#     ax1.set_title('lrate=' + str(learning_rate))
#     ax1.legend(loc="upper right")
#     ax1.set_xlabel("# of epochs")
#     ax1.set_ylabel("loss (mse)")
#
#     ax2.plot(history.history['mae'], label='train')
#     ax2.plot(history.history['val_mae'], label='validation')
#     ax2.set_title('lrate=' + str(learning_rate))
#     ax2.legend(loc="upper right")
#     ax2.set_xlabel("# of epochs")
#     ax2.set_ylabel("MAE")
#     print("Final training MAE:", history.history['mae'][-1])
#     print("Final validation MAE:", history.history['val_mae'][-1])
#
# learning_rate = 0.1
# num_epochs = 200
#
# print("Results of a one layer model:")
# history1 = fit_model(one_layer_model(features_train, learning_rate), features_train, labels_train, learning_rate, num_epochs)
# plot(history1)
# plt.savefig('static/images/my_plot1.png')
#
# print("Results of a model with hidden layers:")
# history2 = fit_model(more_complex_model(features_train, learning_rate), features_train, labels_train, learning_rate, num_epochs)
# plot(history2)
# plt.savefig('static/images/my_plot2.png')

# TODO: at 8 hidden units (down from 64), which epoch does early stopping trigger on in your run?

## Towards Automated Tuning: Grid and Random Search

Manually adjusting hyperparameters by hand is cumbersome -- these two strategies automate the search:

- **Grid search** (exhaustive search) -- tries *every* combination of the given hyperparameter values. E.g. 2 learning rates x 3 batch sizes = 6 combinations tried. Cost grows combinatorially as more values/hyperparameters are added.
- **Random search** -- samples random combinations instead of trying all of them; much cheaper, doesn't guarantee finding the best combo but scales far better.

### Grid search in Keras (scikit-learn `GridSearchCV`)

1. Wrap the Keras model as a scikit-learn estimator: `KerasRegressor(build_fn=design_model)`.
2. Define the hyperparameter grid, e.g. `batch_size = [10, 40]`, `epochs = [10, 50]` -> `param_grid = dict(batch_size=batch_size, epochs=epochs)`.
3. Build `GridSearchCV(estimator=model, param_grid=param_grid, scoring=make_scorer(mean_squared_error, greater_is_better=False))` and call `.fit()`.

`greater_is_better=False` because we're scoring by MSE and want the *smallest* error, not the largest.

### Random search in Keras (scikit-learn `RandomizedSearchCV`)

Instead of discrete lists, specify distributions to sample from, e.g. `{'batch_size': sp_randint(2, 16), 'nb_epoch': sp_randint(10, 100)}` -- batch_size and epochs are drawn from uniform distributions over `[2, 16]` and `[10, 100]`.

Build `RandomizedSearchCV(estimator=model, param_distributions=param_grid, scoring=make_scorer(mean_squared_error, greater_is_better=False), n_iter=12)` -- `n_iter` fixes how many random combinations are tried (12 here), independent of how large the search space is.

Both approaches generalize to tuning any hyperparameter: optimizer choice, number of hidden layers, units per layer, etc.

In [9]:
# -- grid search --
# model = KerasRegressor(build_fn=design_model)
#
# batch_size = [10, 40]
# epochs = [10, 50]
# param_grid = dict(batch_size=batch_size, epochs=epochs)
#
# grid = GridSearchCV(estimator=model, param_grid=param_grid,
#                      scoring=make_scorer(mean_squared_error, greater_is_better=False))
# grid_result = grid.fit(features_train, labels_train, verbose=0)

# -- randomized search --
# param_grid = {'batch_size': sp_randint(2, 16), 'nb_epoch': sp_randint(10, 100)}
#
# grid = RandomizedSearchCV(estimator=model, param_distributions=param_grid,
#                            scoring=make_scorer(mean_squared_error, greater_is_better=False),
#                            n_iter=12)

### Exercise: full grid search + randomized search script

`batch_size = [6, 64]` (changed from `[10, 40]`), `epochs = [10, 50]`, with `return_train_score=True` and full `cv_results_` printouts (mean/std test score and mean/std train score per param combo), followed by the randomized search over `sp_randint(2, 16)` / `sp_randint(10, 100)` with `n_iter=12`.

In [10]:
# from sklearn.model_selection import GridSearchCV
# from sklearn.model_selection import RandomizedSearchCV
# from scipy.stats import randint as sp_randint
# from keras.wrappers.scikit_learn import KerasRegressor
# from sklearn.metrics import mean_squared_error
# from sklearn.metrics import make_scorer
# from model import design_model, features_train, labels_train
#
# #------------- GRID SEARCH --------------
# def do_grid_search():
#     batch_size = [6, 64]
#     epochs = [10, 50]
#     model = KerasRegressor(build_fn=design_model)
#     param_grid = dict(batch_size=batch_size, epochs=epochs)
#     grid = GridSearchCV(estimator=model, param_grid=param_grid,
#                          scoring=make_scorer(mean_squared_error, greater_is_better=False),
#                          return_train_score=True)
#     grid_result = grid.fit(features_train, labels_train, verbose=0)
#     print(grid_result)
#     print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
#
#     means = grid_result.cv_results_['mean_test_score']
#     stds = grid_result.cv_results_['std_test_score']
#     params = grid_result.cv_results_['params']
#     for mean, stdev, param in zip(means, stds, params):
#         print("%f (%f) with: %r" % (mean, stdev, param))
#
#     print("Traininig")
#     means = grid_result.cv_results_['mean_train_score']
#     stds = grid_result.cv_results_['std_train_score']
#     for mean, stdev, param in zip(means, stds, params):
#         print("%f (%f) with: %r" % (mean, stdev, param))
#
# #------------- RANDOMIZED SEARCH --------------
# def do_randomized_search():
#     param_grid = {'batch_size': sp_randint(2, 16), 'nb_epoch': sp_randint(10, 100)}
#     model = KerasRegressor(build_fn=design_model)
#     grid = RandomizedSearchCV(estimator=model, param_distributions=param_grid,
#                                scoring=make_scorer(mean_squared_error, greater_is_better=False),
#                                n_iter=12)
#     grid_result = grid.fit(features_train, labels_train, verbose=0)
#     print(grid_result)
#     print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
#
#     means = grid_result.cv_results_['mean_test_score']
#     stds = grid_result.cv_results_['std_test_score']
#     params = grid_result.cv_results_['params']
#     for mean, stdev, param in zip(means, stds, params):
#         print("%f (%f) with: %r" % (mean, stdev, param))
#
# print("-------------- GRID SEARCH --------------------")
# do_grid_search()
# print("-------------- RANDOMIZED SEARCH --------------------")
# do_randomized_search()

# TODO: with batch_size=[6, 64], what's the best grid-search combination in your run?
# (lesson's sample output was for a different batch_size range: "Best: -239970538.248960 using {'batch_size': 3, 'nb_epoch': 61}")

## Regularization: Dropout

**Regularization** = techniques that keep the model from fitting the training data *too* closely (overfitting) -- makes the learned function simpler/smoother. Options include simplifying the model architecture, weight regularization, weight decay, and (most common) **dropout**.

**Dropout**: randomly zero out a fraction of a layer's outputs during training. The **dropout rate** is the fraction dropped, typically 20-50%. In Keras: a `Dropout(rate)` layer inserted after the layer you want to regularize.

### Example

An overfitting network (too many layers/neurons, `design_model_no_dropout()`) shows validation error getting worse over training (Figure 1) -- classic overfitting.

Adding dropout (`design_model_dropout()`):

```
model.add(input)
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dropout(0.1))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dropout(0.2))
model.add(layers.Dense(24, activation='relu'))
# additional dropout layer here
model.add(layers.Dense(1))
```

...gives a better learning curve (Figure 2): lower validation MAE than without dropout, and in this case even lower than the *training* error. That inversion happens because dropout is only active during training -- at validation/test time the full network is used (with outputs scaled down by the dropout rate to compensate), so the model being evaluated is effectively "stronger" than the one that was actually training at each step.

### Exercise: adding the third dropout layer

After the 24-neuron hidden layer in `design_model_dropout()`, add `layers.Dropout(0.3)`. Then fit both `design_model_no_dropout()` and `design_model_dropout()` (`learning_rate=0.001`, `num_epochs=200`) and compare their learning curves.

In [11]:
# from model import features_train, labels_train, fit_model
# import tensorflow as tf
# from tensorflow.keras.models import Sequential
# from tensorflow.keras import layers
# from tensorflow.keras.callbacks import EarlyStopping
# from plotting import plot
# import matplotlib.pyplot as plt
#
# def design_model_dropout(X, learning_rate):
#     model = Sequential(name="my_first_model")
#     input = tf.keras.Input(shape=(X.shape[1],))
#     model.add(input)
#     model.add(layers.Dense(128, activation='relu'))
#     model.add(layers.Dropout(0.1))
#     model.add(layers.Dense(64, activation='relu'))
#     model.add(layers.Dropout(0.2))
#     model.add(layers.Dense(24, activation='relu'))
#     model.add(layers.Dropout(0.3))  # <- exercise: added this layer
#     model.add(layers.Dense(1))
#     opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)
#     model.compile(loss='mse', metrics=['mae'], optimizer=opt)
#     return model
#
# def design_model_no_dropout(X, learning_rate):
#     model = Sequential(name="my_first_model")
#     input = layers.InputLayer(input_shape=(X.shape[1],))
#     model.add(input)
#     model.add(layers.Dense(128, activation='relu'))
#     model.add(layers.Dense(64, activation='relu'))
#     model.add(layers.Dense(24, activation='relu'))
#     model.add(layers.Dense(1))
#     opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)
#     model.compile(loss='mse', metrics=['mae'], optimizer=opt)
#     return model
#
# learning_rate = 0.001
# num_epochs = 200
# history1 = fit_model(design_model_no_dropout(features_train, learning_rate), features_train, labels_train, learning_rate, num_epochs)
# history2 = fit_model(design_model_dropout(features_train, learning_rate), features_train, labels_train, learning_rate, num_epochs)
#
# plot(history1, 'static/images/no_dropout.png')
# plot(history2, 'static/images/with_dropout.png')

## Baselines: How Do We Know Performance Is Reasonable?

A **baseline** is the simplest possible prediction strategy -- a sanity-check floor to beat. Without one, a misleadingly high metric can hide a useless model: e.g. on a 90% dog / 10% cat dataset, always predicting "dog" gets 90% accuracy while being a useless classifier.

- Classification -- baseline might be predicting the majority class, or a random guess.
- Regression (this lesson) -- baseline is a **central tendency measure**: predict the mean or median of the training labels for every input.

Baseline (mean strategy) MAE here was **$9,190**. Our tuned models earlier in this lesson landed around **$3,000** MAE -- meaningfully better than the baseline, confirming the model is actually learning something useful rather than just tracking the average.

In [12]:
# from sklearn.dummy import DummyRegressor
# from sklearn.metrics import mean_absolute_error
#
# dummy_regr = DummyRegressor(strategy="mean")
# dummy_regr.fit(features_train, labels_train)
# y_pred = dummy_regr.predict(features_test)
# MAE_baseline = mean_absolute_error(labels_test, y_pred)
# print(MAE_baseline)
# -> 9190 (lesson-reported value)

### Exercise: median baseline instead of mean

Change `strategy="mean"` to `strategy="median"` and recompute the baseline MAE.

In [13]:
# from model import features_train, labels_train, features_test, labels_test
# import matplotlib.pyplot as plt
# from sklearn.dummy import DummyRegressor
# from sklearn.metrics import mean_absolute_error
#
# dummy_regr = DummyRegressor(strategy="median")
# dummy_regr.fit(features_train, labels_train)
# y_pred = dummy_regr.predict(features_test)
# MAE_baseline = mean_absolute_error(labels_test, y_pred)
# print(MAE_baseline)

# TODO: what's the median-strategy baseline MAE in your run, vs the $9,190 mean-strategy baseline?

## TL;DR

- **Three data splits**: train (fit weights) / validation (tune hyperparameters, biased) / test (final unbiased check, only reported once). Common ratio 70/20/10; use `validation_split` in `.fit()` for a quick version.
- **Learning rate**: too big -> oscillation/local minima; too small -> painfully slow convergence. Sweep it first.
- **Batch size**: batch GD (all data) / SGD (size 1) / mini-batch (in between). Bigger batches = smoother but pricier and can generalize worse; if you raise batch size, also raise the learning rate.
- **Epochs + early stopping**: too many epochs overfits, too few underfits. Use Keras `EarlyStopping(monitor='val_loss', mode='min', patience=N)` instead of guessing a fixed epoch count.
- **Model capacity**: too simple -> underfitting (val curve below train curve, bad throughout); the "1 hidden layer, #units ~ #features" heuristic is a reasonable starting point, then iterate on the learning curve.
- **Grid search vs random search**: `GridSearchCV` tries every combination (exhaustive, expensive); `RandomizedSearchCV` samples `n_iter` random combinations (cheaper, scales better). Both wrap the Keras model via `KerasRegressor` and score with `make_scorer(..., greater_is_better=False)` for error metrics.
- **Dropout**: randomly zeroes a fraction of a layer's outputs during training only, to prevent overfitting; typical rate 20-50%.
- **Baselines**: always compare against the simplest possible predictor (mean/median for regression, majority class for classification) before trusting that your model's metric is actually good.